# Testing datasets

In [ ]:
import numpy as np
import pandas as pd
import sklearn.compose
import sklearn.preprocessing
import sklearn.model_selection

from lazyfca import LazyFCA

from utils import estimate_quality

In [ ]:
pd.set_option('display.max_columns', None)

## Breast Cancer Wisconsin (Diagnostic)

https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [ ]:
data = pd.read_csv("./datasets/breast_cancer.csv")
data

In [ ]:
data.diagnosis.value_counts()

In [ ]:
data['target'] = data['diagnosis'].map({'B': 0, 'M': 1})
data

In [ ]:
data = data.drop(columns = ['id', 'diagnosis'])

X = data.drop(columns = ["target"])
y = data["target"].to_numpy()
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size = 0.1, stratify = y, random_state = 42
)

numeric = X.columns
categorical = list(set(X_train.columns) - set(numeric))
ct = sklearn.compose.ColumnTransformer(
    transformers = [
        ("numeric", 'passthrough', numeric),
        ("categorical", sklearn.preprocessing.OneHotEncoder(dtype = 'bool'), categorical)
    ]
)
X_train = pd.DataFrame(ct.fit_transform(X_train), columns = ct.get_feature_names_out())
X_test = pd.DataFrame(ct.transform(X_test), columns = ct.get_feature_names_out())

categorical = [ feature for feature in ct.get_feature_names_out() if feature.startswith("categorical__") ]
X_train[categorical] = X_train[categorical].astype(bool)
X_test[categorical] = X_test[categorical].astype(bool)

y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [ ]:
classifier = LazyFCA(
    pos_params=LazyFCA.Params(
        supporters_covered=1,
        supporter_opposer_ratio=1 / 1.75,
    ),
    neg_params=LazyFCA.Params(
        supporters_covered=2,
        supporter_opposer_ratio=3.25,
    ),
    pos_weight=1.0
)
classifier.fit(X_train, y_train)

In [ ]:
y_pred = classifier.predict(X_test)

In [ ]:
base_metrics = estimate_quality(y_pred, y_test)
base_metrics

In [ ]:
import catboost
cb = catboost.CatBoostClassifier(n_estimators=100)
cb.fit(X_train, y_train)
estimate_quality(cb.predict_proba(X_test), y_test)

## Rice dataset

In [ ]:
data = pd.read_csv("./datasets/rice.csv")
data

In [ ]:
data.Class.value_counts()

In [ ]:
data['Class'] = data['Class'].map({'Osmancik': 0, 'Cammeo': 1})
data

In [ ]:
X = data.drop(columns = ["Class"])
y = data["Class"].to_numpy()
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    X, y, test_size = 0.1, stratify = y, random_state = 42
)

numeric = X.columns
categorical = list(set(X_train.columns) - set(numeric))
ct = sklearn.compose.ColumnTransformer(
    transformers = [
        ("numeric", 'passthrough', numeric),
        ("categorical", sklearn.preprocessing.OneHotEncoder(dtype = 'bool'), categorical)
    ]
)
X_train = pd.DataFrame(ct.fit_transform(X_train), columns = ct.get_feature_names_out())
X_test = pd.DataFrame(ct.transform(X_test), columns = ct.get_feature_names_out())

categorical = [ feature for feature in ct.get_feature_names_out() if feature.startswith("categorical__") ]
X_train[categorical] = X_train[categorical].astype(bool)
X_test[categorical] = X_test[categorical].astype(bool)

y_train = pd.Series(y_train)
y_test = pd.Series(y_test)

In [ ]:
classifier = LazyFCA(
    pos_params=LazyFCA.Params(
        supporters_covered=3,
        supporter_opposer_ratio=1 / 1.0,
    ),
    neg_params=LazyFCA.Params(
        supporters_covered=5,
        supporter_opposer_ratio=1.0,
    ),
    pos_weight=1.0
)
classifier.fit(X_train, y_train)

In [53]:
y_pred = classifier.predict(X_test)

100%|██████████| 381/381 [00:12<00:00, 31.21it/s]


In [54]:
base_metrics = estimate_quality(y_pred, y_test)
base_metrics

{'Accuracy': 0.7480314960629921,
 'Precision': 0.9240506329113924,
 'Recall': 0.44785276073619634,
 'AUC-ROC': 0.8974503292621152,
 'F1-score': 0.6033057851239669,
 'True Positive': 73,
 'True Negative': 212,
 'False Positive': 6,
 'False Negative': 90,
 'True Negative Rate (Specificity)': 0.9724770642201835,
 'Negative Predictive Value': 0.7019867549668874,
 'False Positive Rate': 0.027522935779816515,
 'False Discovery Rate': 0.0759493670886076}

In [56]:
import catboost
cb = catboost.CatBoostClassifier(n_estimators=2500)
cb.fit(X_train, y_train)
estimate_quality(cb.predict_proba(X_test), y_test)

Learning rate set to 0.007526
0:	learn: 0.6806184	total: 2.35ms	remaining: 5.88s
1:	learn: 0.6709301	total: 5.14ms	remaining: 6.42s
2:	learn: 0.6596439	total: 8.02ms	remaining: 6.67s
3:	learn: 0.6496490	total: 10.5ms	remaining: 6.55s
4:	learn: 0.6394989	total: 12.8ms	remaining: 6.39s
5:	learn: 0.6286072	total: 15.1ms	remaining: 6.28s
6:	learn: 0.6182668	total: 17.4ms	remaining: 6.21s
7:	learn: 0.6097360	total: 19.4ms	remaining: 6.05s
8:	learn: 0.6001001	total: 21.4ms	remaining: 5.93s
9:	learn: 0.5909345	total: 24ms	remaining: 5.97s
10:	learn: 0.5824652	total: 25.9ms	remaining: 5.87s
11:	learn: 0.5724534	total: 27.8ms	remaining: 5.77s
12:	learn: 0.5634124	total: 30.1ms	remaining: 5.76s
13:	learn: 0.5553876	total: 32.1ms	remaining: 5.7s
14:	learn: 0.5474499	total: 34ms	remaining: 5.64s
15:	learn: 0.5400554	total: 36.2ms	remaining: 5.62s
16:	learn: 0.5308499	total: 38.1ms	remaining: 5.57s
17:	learn: 0.5231953	total: 40ms	remaining: 5.52s
18:	learn: 0.5159988	total: 42.2ms	remaining: 5.5s


{'Accuracy': 0.9212598425196851,
 'Precision': 0.8982035928143712,
 'Recall': 0.9202453987730062,
 'AUC-ROC': 0.9770923622446108,
 'F1-score': 0.9090909090909091,
 'True Positive': 150,
 'True Negative': 201,
 'False Positive': 17,
 'False Negative': 13,
 'True Negative Rate (Specificity)': 0.9220183486238532,
 'Negative Predictive Value': 0.9392523364485982,
 'False Positive Rate': 0.0779816513761468,
 'False Discovery Rate': 0.10179640718562874}